# GeoFWI3D — find three qualifying baseline seeds across three folds

**Completion means 3 seeds × 3 folds × 4 models = 36 verified passing results.**
Each seed must achieve **strictly greater than 50%** final-budget validation foreground Dice in
**every one of its 12 model/fold cells**. Exactly 50% fails. No rounding or averaging is used for selection.
Seeds are tried in numerical order, starting at 0. On the first completed failing cell, reject the entire
seed, retain its evidence, skip its remaining cells and continue. Technical errors pause the search;
they do not reject a seed. Only the first three qualifying seeds are selected.

- GeoFWI3D native 96³ seismic images, physical salt masks, frozen folds 0/1/2.
- 19 training and 96 validation volumes per fold; the same subsets for every model/seed.
- **No augmentation**, batch 1, BF16 autocast with FP32 parameters/optimizer, training from scratch.
- **182,178.12654316 GFLOPs per run**; final budget checkpoint is primary; best validation is secondary.
- nnU-Net: native foreground Dice+CE, deep supervision and SGD, **151 updates**.
- SwinUNETR: Dice and AdamW, **362 updates**; UNETR: **414**; 3D U-Net width 16: **952**.
- Original learning-rate trajectories stay fixed. Preflight, validation, I/O and preprocessing are outside
  the optimization cap and reported separately. There are no constraints or auxiliary calibration.

**This is a validation-selected seed cohort.** Keep all attempted/rejected scores when reporting it;
it is not an unbiased random-seed estimate or an independent test. Success is not guaranteed at this cap.
The notebook never marks a session limit, technical failure, or insufficient passing seeds as complete.

Select an **A100 GPU and High RAM**, then run all cells. Inputs are already uploaded to
`MyDrive/hippo_geofwi3d_baseline_seed_search/input` on **filippofocaccia2@gmail.com**.
No SSH credentials, cluster access or pretrained model downloads are required.
Run only one copy of this notebook against this results folder.


In [1]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from googleapiclient.discovery import build
import google.auth
credentials, _ = google.auth.default()
service = build('drive','v3',credentials=credentials,cache_discovery=False)
about = service.about().get(fields='user(emailAddress),storageQuota').execute()
assert about['user']['emailAddress'].lower() == 'filippofocaccia2@gmail.com', 'Mount/authenticate the requested account.'
quota=about['storageQuota']
if 'limit' in quota:
    assert int(quota['limit'])-int(quota['usage']) >= 8*1024**3, 'Keep at least 8 GiB free in Drive, including trash.'
from pathlib import Path
import hashlib,json,os,shutil,signal,subprocess,sys,time,zipfile
HOME_DIR=Path('/content/drive/MyDrive/hippo_geofwi3d_baseline_seed_search')
INPUT=HOME_DIR/'input'
RESULTS=HOME_DIR/'results_mixed_noaug_cap182178_v1'
BUNDLE=Path('/content/hippo_geofwi_seed_bundle')
RESULTS.mkdir(parents=True,exist_ok=True)
BUNDLE.mkdir(exist_ok=True)
assert shutil.disk_usage('/content').free >= 20*1024**3, 'Need at least 20 GiB runtime disk space.'
print('Account:',about['user']['emailAddress'],'Results:',RESULTS)


Mounted at /content/drive


Account: filippofocaccia2@gmail.com Results: /content/drive/MyDrive/hippo_geofwi3d_baseline_seed_search/results_mixed_noaug_cap182178_v1


## Verify and stage the uploaded data/code
All archive and per-file SHA256 hashes are checked. The 319 unique images/labels are read from
the runtime SSD during training. Existing matching files are reused; changed files stop execution.


In [2]:
EXPECTED_UPLOADS={'geofwi3d_seed_search_code.zip': '1c63644da6739c431e2e2b074fbe60e417bbdf55c6acd12dca70464bed12b381', 'geofwi3d_data_part01.zip': '32eaadf1a6abd61dd5bd922a8a085af4e6c876b34b915bbd09da93a3dbea2ca2', 'geofwi3d_data_part02.zip': '31adc6fa680b74a921f9457f24d2dfbb178aa1cef1d80baa152c5ee82a4e05bd', 'geofwi3d_data_part03.zip': 'e9c404631c47a918d1ca36c9166ea92141aa1b8264e332fbaa32b13299d8706a', 'geofwi3d_data_part04.zip': '1ef1e965ebefb571f441d986f2304edb4f2ab824b4bedb7dfbc6a494c4c1c279', 'geofwi3d_data_part05.zip': 'b99cbe88932b94bc382512c059661b48516beacd1dae116237ec411edc64fb58', 'geofwi3d_data_part06.zip': '6f0509326647688300459b05d5f0af5c10ecdde9c19fc6fc3ee05bc92fe8f183', 'geofwi3d_data_part07.zip': '3e1cf8a1cacd445c73bf1798ebd9c1b3c34f6f2cea0e2bf0ad378c6f12f09fb1', 'geofwi3d_data_part08.zip': 'b2b603f5ef55eb2822e64904ce7d178b7cd459c1258398de1f75cdedad5525ed', 'geofwi3d_data_part09.zip': '833fccd13996d6d6ddd118b6146334a902e4eceba2dedd26b798db64fcf6de38', 'geofwi3d_data_part10.zip': '99daf1d8f684ac47897aaa6933b69741e826935783bc42dbd02093efc299e235', 'geofwi3d_data_part11.zip': 'ce12340263414416fcd728e7fad43a2dbbbf4d26f22cc51273acb8df50680239', 'geofwi3d_data_part12.zip': 'ff0f922b1ae82afb2394725bcd0374b4eb273fe0333f5fbad4dbd52b540aadd2', 'geofwi3d_data_part13.zip': '9ede65d6da572c0186b51b838f46f7cd281bbd8b515aef813e502b59891cc880'}
EXPECTED_MANIFEST='7e539f89f55137ebd30830c105ded0c0b56ba06710460768c60a4a5aed12a9ea'

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()
for name,expected in EXPECTED_UPLOADS.items():
    package=INPUT/name
    if not package.exists():raise FileNotFoundError(str(package))
    assert sha(package)==expected, 'Upload checksum mismatch: '+name
    local=Path('/content')/name
    if not local.exists() or sha(local)!=expected:
        shutil.copyfile(package,local)
    assert sha(local)==expected
    with zipfile.ZipFile(local) as z:
        assert len(z.namelist())==len(set(z.namelist())), 'Duplicate archive members'
        for info in z.infolist():
            p=Path(info.filename)
            assert not p.is_absolute() and '..' not in p.parts
            assert (info.external_attr>>16)&0o170000 != 0o120000
        for info in z.infolist():
            target=BUNDLE/info.filename
            if target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            temp=target.with_name(target.name+'.extracting')
            with z.open(info) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst)
            temp.replace(target)
    print('Staged',name)
assert sha(BUNDLE/'bundle_manifest.json')==EXPECTED_MANIFEST
for listing in ['bundle_manifest.json','data_manifest.json']:
    for name,expected in json.loads((BUNDLE/listing).read_text()).items():
        assert sha(BUNDLE/name)==expected, 'Extracted file mismatch: '+name
source=RESULTS/'provenance';source.mkdir(exist_ok=True)
for name in ['geofwi3d_seed_search_code.zip']:
    dst=source/name
    if not dst.exists():shutil.copyfile(INPUT/name,dst)
    assert sha(dst)==EXPECTED_UPLOADS[name]
print('All inputs verified. No augmentation.')


Staged geofwi3d_seed_search_code.zip
Staged geofwi3d_data_part01.zip
Staged geofwi3d_data_part02.zip
Staged geofwi3d_data_part03.zip
Staged geofwi3d_data_part04.zip
Staged geofwi3d_data_part05.zip
Staged geofwi3d_data_part06.zip
Staged geofwi3d_data_part07.zip
Staged geofwi3d_data_part08.zip
Staged geofwi3d_data_part09.zip
Staged geofwi3d_data_part10.zip
Staged geofwi3d_data_part11.zip
Staged geofwi3d_data_part12.zip
Staged geofwi3d_data_part13.zip
All inputs verified. No augmentation.


## Install the frozen environment
Uses the package versions from the existing Colab training environment, in a separate venv.
Setup output is streamed and saved in Drive. Keep the same GPU type/software on resume.


In [3]:
VENV=Path('/content/hippo_geofwi_seed_venv');PYTHON=VENV/'bin/python'
def command(args,log_name):
    with (RESULTS/log_name).open('a') as log:
        proc=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,bufsize=1,start_new_session=True)
        tail=[]
        def drain():
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
                tail.append(line);del tail[:-40]
        try:drain();code=proc.wait()
        except KeyboardInterrupt:
            # Signal the orchestrator only; it stops the active trainer at a safe boundary.
            proc.send_signal(signal.SIGINT)
            print('Saving recovery to Drive. Please wait; do not interrupt again.')
            drain();code=proc.wait()
        finally:proc.stdout.close()
        if code:raise RuntimeError('Process failed; recovery retained.\n'+''.join(tail))
if not PYTHON.exists():
    command([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup.log')
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    command([PYTHON,'-m','ensurepip','--upgrade'],'setup.log')
command([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'],'setup.log')
def campaign(action,*extra):
    command([PYTHON,'-u',BUNDLE/'runtime.py',action,'--bundle',BUNDLE,'--durable',RESULTS,*extra],action+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.2 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CompletedProcess(args=['nvidia-smi'], returncode=0)

## Required GPU preflight
All four models must pass a real native-96³ BF16 forward/backward, finite-gradient check,
binary confusion-count evaluation and exact model/optimizer/RNG checkpoint replay before screening.
These diagnostic updates are saved as preflight evidence, never scientific results.


In [4]:
campaign('preflight')


I0000 00:00:1792338603.881382    2151 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792338603.954625    2151 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792338606.796072    2151 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792338628.685777    2337 port.cc:153] oneDNN custom operations are on. You may see slightly different numerica

RuntimeError: Process failed; recovery retained.
  File "/content/hippo_geofwi_seed_bundle/worker.py", line 83, in equal
    if isinstance(a,dict):return a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)
                                                        ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/hippo_geofwi_seed_bundle/worker.py", line 83, in <genexpr>
    if isinstance(a,dict):return a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)
                                                            ~~~~~^^^^^^^^^^^
  File "/content/hippo_geofwi_seed_bundle/worker.py", line 83, in equal
    if isinstance(a,dict):return a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)
                                                        ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/hippo_geofwi_seed_bundle/worker.py", line 83, in <genexpr>
    if isinstance(a,dict):return a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)
                                            

## Run or resume the search
The job continues until three seeds qualify. A session timer pauses safely; rerun this cell or
the notebook in a new runtime to continue. Completed and rejected seeds are reconstructed from
hash-verified archives. Already completed experiments are never rerun automatically.

Interrupt **once** to request a graceful checkpoint and wait for the Drive upload. A hard VM loss
can occur between the latest recovery snapshot and later charged updates; in that case the notebook
stops for a compute/recovery audit instead of silently losing updates, exceeding the cap, or discarding
that seed. No automatic restart can guarantee recovery of work that Colab destroyed before upload.


In [ ]:
SESSION_HOURS=10.0  # Conservative clean-stop target; increase only within your session allowance.
MAX_NEW_RUNS_THIS_SESSION=10000  # A session guard, never a scientific completion rule.
# Creating RESULTS/'STOP' requests a stop. Remove it only when intentionally resuming.
campaign('run','--hours',str(SESSION_HOURS),'--max-runs',str(MAX_NEW_RUNS_THIS_SESSION))


## Inspect results and the completion marker
`search_status.json` records qualified/rejected seeds and reasons. `all_attempted_results.csv`
preserves every completed cell. `qualified_baselines.csv` contains the selected cohort.
`COMPLETE.json` exists only after all **36** selected cells have final validation Dice **>50%**.
Per-run histories, final/best metrics, FN/FP counts, configuration, seed/fold, initialization hashes,
FLOPs ledger, source provenance and logs are preserved. Checkpoints are removed only after verified
archival. The input data and source archives remain in Drive for future paired constraint experiments.


In [ ]:
campaign('report')
import pandas as pd
status=json.loads((RESULTS/'search_status.json').read_text())
print('Status:',status['status'],'Qualified seeds:',status['qualified_seeds'])
display(pd.DataFrame(status['decisions']))
if status['runs']:
    table=pd.DataFrame(status['runs'])
    table['validation_dice_percent']=100*table['final_dice']
    display(table.pivot(index=['seed','fold'],columns='model',values='validation_dice_percent'))
if (RESULTS/'COMPLETE.json').exists():
    print('COMPLETE:',json.loads((RESULTS/'COMPLETE.json').read_text()))
else:print('Not complete: resume the run cell when ready.')
